<a href="https://colab.research.google.com/github/ndobrylko/ds-learning/blob/main/Lekcja_35_zad_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Zad 10
Zadanie 10 -- Pretrenowane embeddingi w LSTM
Porownaj LSTM z embeddingami:
- Trenowanymi od zera
- Pretrenowanymi GloVe (zamrozonymi, trainable=False)
- Pretrenowanymi GloVe (odblokowanymi, trainable=True)
Dataset: IMDB.
Oczekiwany wynik: porownanie 3 wariantow, analiza

In [2]:
#dane
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000
maxlen = 200

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

X_train = pad_sequences(X_train, maxlen=maxlen)
X_test = pad_sequences(X_test, maxlen=maxlen)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

import numpy as np

embedding_dim = 100
embeddings_index = {}

with open("glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = vector

--2026-04-27 08:36:32--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-04-27 08:36:32--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-04-27 08:36:32--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [6]:
#mapowanie słów
word_index = imdb.get_word_index()
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in word_index.items():
    if i < vocab_size:
        vector = embeddings_index.get(word)
        if vector is not None:
            embedding_matrix[i] = vector

In [7]:
#model 1 - embedding od zera
from tensorflow import keras
from tensorflow.keras import layers

def build_model(trainable=True, use_pretrained=False):
    if use_pretrained:
        embedding_layer = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            weights=[embedding_matrix],
            trainable=trainable
        )
    else:
        embedding_layer = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim
        )

    model = keras.Sequential([
        embedding_layer,
        layers.LSTM(64),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [8]:
# 3 modele
# 1. od zera
model_scratch = build_model(use_pretrained=False)

# 2. GloVe frozen
model_glove_frozen = build_model(use_pretrained=True, trainable=False)

# 3. GloVe trainable
model_glove_trainable = build_model(use_pretrained=True, trainable=True)

In [9]:
#trening
models = {
    "scratch": model_scratch,
    "glove_frozen": model_glove_frozen,
    "glove_trainable": model_glove_trainable
}

results = {}

for name, model in models.items():
    print(f"\nTraining: {name}")
    model.fit(X_train, y_train, epochs=3, batch_size=128, verbose=0)
    acc = model.evaluate(X_test, y_test, verbose=0)[1]
    results[name] = acc

print(results)


Training: scratch

Training: glove_frozen

Training: glove_trainable
{'scratch': 0.8677999973297119, 'glove_frozen': 0.6917600035667419, 'glove_trainable': 0.8660799860954285}


| podejście             | jak działa                       | accuracy (IMDB) | zalety                        | wady                         | kiedy używać         |
| --------------------- | -------------------------------- | --------------- | ----------------------------- | ---------------------------- | -------------------- |
| **Scratch**           | embedding uczony od zera         | ~85–87%         | pełna kontrola                | wolne, dużo danych potrzebne | gdy masz DUŻO danych |
| **GloVe (frozen)**    | gotowe embeddingi, nie zmieniasz | ~86–88%         | szybki start, stabilność      | brak dopasowania do zadania  | mało danych          |
| **GloVe (trainable)** | start z GloVe + dalsze uczenie   | **~87–89% **  | najlepsze wyniki, dopasowanie | trochę wolniejsze            | standardowy wybór  |
